# Problem 2

In [1]:
import gurobipy as gp
from gurobipy import GRB

# --------------------------
# 1. INPUT DATA
# --------------------------
stations = [1, 2, 3, 4, 5, 6, 7, 8]

# OD Demand Matrix (8x8): demand_data[i-1][j-1] = # of passengers i->j
demand_data = [
    [0,   150, 300, 200, 100, 250, 180, 220],  # from station 1
    [160, 0,   210, 270, 130, 190, 240, 150],  # from station 2
    [240, 180, 0,   310, 140, 260, 170, 200],  # from station 3
    [200, 220, 190, 0,   280, 210, 320, 160],  # from station 4
    [130, 140, 150, 160, 0,   230, 250, 290],  # from station 5
    [210, 170, 260, 230, 310, 0,   180, 150],  # from station 6
    [190, 240, 200, 270, 220, 160, 0,   210],  # from station 7
    [250, 200, 150, 180, 210, 230, 190, 0  ]   # from station 8
]

# Travel-time matrix (minutes): time_data[i-1][j-1] = travel time i->j
time_data = [
    [0,  10, 15, 20, 25, 30, 35, 40],  # from station 1
    [12, 0,  18, 24, 16, 20, 28, 32],  # from station 2
    [14, 16, 0,  22, 26, 30, 18, 24],  # from station 3
    [20, 24, 26, 0,  12, 18, 22, 28],  # from station 4
    [18, 14, 20, 16, 0,  15, 25, 30],  # from station 5
    [25, 20, 22, 28, 30, 0,  14, 19],  # from station 6
    [30, 28, 24, 20, 18, 15, 0,  17],  # from station 7
    [35, 32, 28, 24, 20, 18, 16, 0 ]   # from station 8
]

# Convert Time Travel and OD matrices into dictionaries for easy lookup.
ArcDemand = {}       #OD Arcs and passenger demand 
ArcTime = {}      #Time between Arcs

# Loop through the 8x8 staion matrix and create arc time and demand
for i in stations:      
    for j in stations:
        ArcDemand[(i, j)] = demand_data[i-1][j-1]  #Arc demand
        ArcTime[(i, j)] = time_data[i-1][j-1]   #Arc time

# Problem parameters
bus_capacity   = 150
max_buses      = 5
total_time_cap = 391890
fixed_cost     = 1000
station_cost   = 100

# --------------------------
# 2. ENUMERATE ROUTES (ASCENDING AND DESCENDING)
# --------------------------

# Generate all no backtracking routes from i to j store in "all_routes"
all_routes = []
for i in stations:  #loop through all Stations
    for j in stations:
        if i != j:  #Ignore when stations are the same
            # Determine direction by finding out lower and higher station values.
            EarlierStation = min(i, j)
            LasterStation = max(i, j)
            intermediate = []
            #sort through lowest number to highest number station within this route
            for m in stations:
                if m > EarlierStation and m < LasterStation:       #if station is within this route
                    intermediate.append(m)  #add station to the route

            # In order to loop through all possible station combinations within this route
            # There are 2^(number of intermediates) subsets of intermediate stations
            n_mid = len(intermediate)       # Identify total stations within the route
            num_subsets = 1 << n_mid        # bit shift to the left, this is the same as taking 2^n_mid
            
            subset_mask = 0
            
            # Loop through all possible intermediate stations within the low and high range
            while subset_mask < num_subsets:
                chosen = []
                
                # Add starting station
                chosen.append(i)
                
                # Add mid section intermediate stations.
                for idx in range(n_mid):
                    bit = 1 << idx
                    if (subset_mask & bit) != 0:
                        chosen.append(intermediate[idx])
                
                # Add last ending station
                chosen.append(j)  
                
                # Sort chosen stations direction
                if i < j:
                    #if going forward
                    chosen.sort()            # ascending order
                else:
                    #else sort it by reverse direction
                    chosen.sort(reverse=True)  # descending order
                
                # Save the route as a tuple.
                all_routes.append(tuple(chosen))
                subset_mask += 1

# Since list can contain duplicates Remove duplicates by forcing as set
all_routes = list(set(all_routes))
all_routes.sort()           # Sort the list 
R = range(len(all_routes))  # Total Routes

# --------------------------
# 3. PRECOMPUTE ROUTE COST, ROUTE LENGTH & ROUTE TRAVEL TIMES
# --------------------------
route_cost = {}     #Route Cost
route_length = {}   #Route Length
route_time = {}     #Route Travel Time in minutes for OD pair (i, j) on route r
                    #route_time[((i, j), r)] 

# R has total routes, and we will go through each route r_ind
for r_idx in R:
    # Update Route Cost
    route_stations = all_routes[r_idx]      # Identify all stations in this route
    length_r = len(route_stations)          # Count total stations in this route
    route_cost[r_idx] = fixed_cost + station_cost * length_r# Calculate cost for this route
    
    # Update Route Length
    route_length[r_idx] = length_r
    
    # Update Route Travel Time
    cumulative_station_time = {}
    first_station = route_stations[0]             # First station
    cumulative_station_time[first_station] = 0            # First station as start time
    total_time = 0
    
    # Loop through each arc and create an initial cumulative time for each station
    for idx in range(len(route_stations) - 1):
        s1 = route_stations[idx]                # Current station
        s2 = route_stations[idx + 1]            # Next station
        # Look up ArcTime dictionary of time between current station and next station
        total_time += ArcTime[(s1, s2)]         # Add up time between two stations
        cumulative_station_time[s2] = total_time

    # Loop through each arc and update route travel time
    for (i_val, j_val) in ArcDemand.keys():
        # Continue if arc isn't looping to itself (e.g. 1,1)
        if i_val != j_val:
            if i_val in route_stations and j_val in route_stations: #If this arc is within this route
                idx_i = route_stations.index(i_val)
                idx_j = route_stations.index(j_val)
                if idx_i < idx_j:                                   #Figure out direction
                    # If station direction is ascending then update route time
                    route_time[((i_val, j_val), r_idx)] = cumulative_station_time[j_val] - cumulative_station_time[i_val]
                else:
                    # Else in reverse and set to zero
                    route_time[((i_val, j_val), r_idx)] = 0
            else:
                # If this arc isn't within this route then set to zero time
                route_time[((i_val, j_val), r_idx)] = 0
        else:
            # If this arc is self looping then set to zero time
            route_time[((i_val, j_val), r_idx)] = 0

# --------------------------
# 4. BUILD THE MODEL
# --------------------------
model = gp.Model("BiDirectional_Route_Model")

# Decision variables:
y_vars = {} # y_vars[r]: binary variable indicating whether route r is used.
b_vars = {} # b_vars[r]: integer number of buses assigned to route r (0 to max_buses).
x_vars = {} # x_vars[(i,j), r]: continuous variable representing passenger flow for OD (i,j) on route r.

# R is total number of all routs. Value from all_routes variable length
for r_idx in R:
    #y tracks if each route are used or not
    y_vars[r_idx] = model.addVar(vtype=GRB.BINARY, name="y_r%d" % r_idx)
    #b sets Bus quantity with max bus as upper bound
    b_vars[r_idx] = model.addVar(vtype=GRB.INTEGER, lb=0, ub=max_buses, name="b_r%d" % r_idx)

# K is a list of OD arc pairs (i, j) for reference
K = []
for i in stations: # Start from station 1 to 8
    for j in stations:
        if i != j: # Make sure not self looping arc
            K.append((i, j)) # Add arc pairs to K

# Create arc demand x_var for each OD arc pairs            
for (i_val, j_val) in K:
    # Loop through all routs and add variables 
    for r_idx in R:
        # initialize x variable for all OD arc demand
        x_vars[((i_val, j_val), r_idx)] = model.addVar(vtype=GRB.CONTINUOUS, lb=0,
                                                        name="x_%d_%d_r%d" % (i_val, j_val, r_idx))

# --------------------------
# 5. SET THE OBJECTIVE
# --------------------------
# Minimize the total cost of used routes.
obj_expr = gp.LinExpr() # Create linear expression objective

# Loop through all routes and create an objective function with term of Route Cost * y variable (each route)
for r_idx in R:
    obj_expr.addTerms(route_cost[r_idx], y_vars[r_idx])

# Create objective function to minimization Route cost
model.setObjective(obj_expr, GRB.MINIMIZE)

# --------------------------
# 6. ADD CONSTRAINTS
# --------------------------
# (a) Demand satisfaction: For every OD pair, the sum over routes must equal the demand.
for (i_val, j_val) in K:    # Loop through each OD arc pairs K(i, j)
    lhs = gp.LinExpr()      # Create linear expression
    for r_idx in R:         # Loop through all routs and add constraints 
        lhs.addTerms(1.0, x_vars[((i_val, j_val), r_idx)])
    # Set x variable to all OD arc demand
    model.addConstr(lhs == ArcDemand[(i_val, j_val)], "demand_%d_%d" % (i_val, j_val))

# (b) Capacity constraint: Passenger flow on route r cannot exceed bus capacity times number of buses.
for r_idx in R:             # Loop through all routs and add constraints 
    lhs_cap = gp.LinExpr()  # Create linear expression objective
    for (i_val, j_val) in K:# Loop through each OD arc pairs K(i, j)
        lhs_cap.addTerms(1.0, x_vars[((i_val, j_val), r_idx)])
    # All x <= bus capacity * bus quantity
    model.addConstr(lhs_cap <= bus_capacity * b_vars[r_idx], "cap_r%d" % r_idx)

# (c) Bus-route activation: Number of buses is zero if route is not used.
for r_idx in R: # Loop through all routs and add constraints 
    model.addConstr(b_vars[r_idx] <= max_buses * y_vars[r_idx], "buslink_r%d" % r_idx)

# (d) Total travel time constraint.
lhs_time = gp.LinExpr()     # Create linear expression objective
for (i_val, j_val) in K:    # Loop through each OD arc pairs K(i, j)
    for r_idx in R:         # Loop through all routs and add constraints 
        val_time = route_time[((i_val, j_val), r_idx)]
        if val_time > 0:
            lhs_time.addTerms(val_time, x_vars[((i_val, j_val), r_idx)])
# all rout time <= total time cap of 391890
model.addConstr(lhs_time <= total_time_cap, "time_limit")

# (e) Force x_vars to zero if the route does not serve an OD pair.
for (i_val, j_val) in K:    # Loop through each OD arc pairs K(i, j)
    for r_idx in R:         # Loop through all routs and add constraints 
        if route_time[((i_val, j_val), r_idx)] == 0:    #If route time is zero then set to zero
            model.addConstr(x_vars[((i_val, j_val), r_idx)] == 0,
                            "no_service_%d_%d_r%d" % (i_val, j_val, r_idx))

# --------------------------
# 7. SOLVE THE MODEL
# --------------------------
modelTimeLimit = 0.2                            # Limit Gurobi optimization processing time e.g. 1 is 1min time limit
model.setParam('TimeLimit', modelTimeLimit*60)  # Set model time limit
model.optimize()

# --------------------------
# 8. DISPLAY RESULTS AND ROUTES
# --------------------------
# Print objective min cost
print("Objective (min cost) =", model.ObjVal)
used_routes = []
for r_idx in R:
    if y_vars[r_idx].X > 0.5:
        used_routes.append(r_idx)

 # Display route used.
print("Number of routes used =", len(used_routes))
for r_idx in used_routes:
    route_stations = all_routes[r_idx]
    print("  Route index =", r_idx, "stations =", route_stations)
    print("    y =", y_vars[r_idx].X)
    print("    buses =", b_vars[r_idx].X)
    print("    route cost =", route_cost[r_idx])

 # Calculate total passenger travel time used.
total_time_used = 0.0
for (i_val, j_val) in K:
    for r_idx in R:
        flow = x_vars[((i_val, j_val), r_idx)].X
        if flow > 1e-6:
             total_time_used += route_time[((i_val, j_val), r_idx)] * flow
print("Total passenger travel time =", total_time_used, "(limit =", total_time_cap, ")")


Set parameter Username
Academic license - for non-commercial use only - expires 2025-08-19
Set parameter TimeLimit to value 12
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 25125 rows, 28652 columns and 84474 nonzeros
Model fingerprint: 0xf09c17fb
Variable types: 27664 continuous, 988 integer (494 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [1e+03, 2e+03]
  Bounds range     [1e+00, 5e+00]
  RHS range        [1e+02, 4e+05]
Presolve removed 24574 rows and 24574 columns
Presolve time: 0.02s
Presolved: 551 rows, 4078 columns, 9704 nonzeros
Variable types: 3584 continuous, 494 integer (494 binary)
Found heuristic solution: objective 71800.000000

Root relaxation: objective 2.085089e+04, 1195 iterations, 0.01 seconds (0.01 work units)

    Nodes    |    Current Node    |     Objective Bounds     

# Problem 3

In [5]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model
model = gp.Model("BasicLP")

# Create variable
# x1: number of units of Prod1
# x2: number of units of Prod2
# x3: number of units of Prod3
x1 = model.addVar(lb=15.3, ub=102.3, vtype=GRB.CONTINUOUS, name="x1")
x2 = model.addVar(lb=23.2, ub=357.4,  vtype=GRB.CONTINUOUS, name="x2")
x3 = model.addVar(lb=17.5, ub=182.6,  vtype=GRB.CONTINUOUS, name="x3") 


# Set objective function: Maximize total profit
model.setObjective(20*x1 + 40*x2 + 32*x3, GRB.MAXIMIZE)

# Add processing time constraint:
model.addConstr(4.6*x1 + 7.5*x2 + 2.4*x3 <= 556.7, name="ProcessingTime")

# We already used lower/upper bounds to enforce minimum and maximum production.
# Non-negativity is automatically enforced since lb >= 0.

# Optimize the model
model.optimize()

# Check if an optimal solution was found and print results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(" x1 =", x1.X)
    print(" x2 =", x2.X)
    print(" x3 =", x3.X)
    print(" Objective value =", model.ObjVal)
else:
    print("No optimal solution found.")


Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 1 rows, 3 columns and 3 nonzeros
Model fingerprint: 0xac175d7e
Coefficient statistics:
  Matrix range     [2e+00, 8e+00]
  Objective range  [2e+01, 4e+01]
  Bounds range     [2e+01, 4e+02]
  RHS range        [6e+02, 6e+02]
Presolve removed 1 rows and 3 columns
Presolve time: 0.02s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.3982667e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.03 seconds (0.00 work units)
Optimal objective  5.398266667e+03
Optimal solution found:
 x1 = 15.3
 x2 = 23.2
 x3 = 130.13333333333335
 Objective value = 5398.266666666667


### branch 1: 130<x3<130 

In [9]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model
model = gp.Model("BasicLP")

# Create variables (continuous, with given bounds)
# x1: number of units of Prod1
# x2: number of units of Prod2
# x3: number of units of Prod3
x1 = model.addVar(lb=15.3, ub=102.3, vtype=GRB.CONTINUOUS, name="x1")
x2 = model.addVar(lb=23.2, ub=357.4,  vtype=GRB.CONTINUOUS, name="x2")
x3 = model.addVar(lb=17.5, ub=182.6,  vtype=GRB.CONTINUOUS, name="x3") 


# Set objective function: Maximize total profit
model.setObjective(20*x1 + 40*x2 + 32*x3, GRB.MAXIMIZE)

# Add processing time constraint:
model.addConstr(4.6*x1 + 7.5*x2 + 2.4*x3 <= 556.7, name="ProcessingTime")
model.addConstr(x3 <= 130, name = 'branch 1')


# Optimize the model
model.optimize()

# Check if an optimal solution was found and print results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(" x1 =", x1.X)
    print(" x2 =", x2.X)
    print(" x3 =", x3.X)
    print(" Objective value =", model.ObjVal)
else:
    print("No optimal solution found.")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 2 rows, 3 columns and 4 nonzeros
Model fingerprint: 0x45de678f
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+01, 4e+01]
  Bounds range     [2e+01, 4e+02]
  RHS range        [1e+02, 6e+02]
Presolve removed 2 rows and 3 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.3957067e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  5.395706667e+03
Optimal solution found:
 x1 = 15.3
 x2 = 23.242666666666672
 x3 = 130.0
 Objective value = 5395.706666666667


In [20]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model
model = gp.Model("BasicLP")

# Create variables (continuous, with given bounds)
# x1: number of units of Prod1
# x2: number of units of Prod2
# x3: number of units of Prod3
x1 = model.addVar(lb=15.3, ub=102.3, vtype=GRB.CONTINUOUS, name="x1")
x2 = model.addVar(lb=23.2, ub=357.4,  vtype=GRB.CONTINUOUS, name="x2")
x3 = model.addVar(lb=17.5, ub=182.6,  vtype=GRB.CONTINUOUS, name="x3") 


# Set objective function: Maximize total profit
model.setObjective(20*x1 + 40*x2 + 32*x3, GRB.MAXIMIZE)

# Add processing time constraint:
model.addConstr(4.6*x1 + 7.5*x2 + 2.4*x3 <= 556.7, name="ProcessingTime")
model.addConstr(x3 >= 131, name = 'branch 1')


# Optimize the model
model.optimize()

# Check if an optimal solution was found and print results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(" x1 =", x1.X)
    print(" x2 =", x2.X)
    print(" x3 =", x3.X)
    print(" Objective value =", model.ObjVal)
else:
    print("No optimal solution found.")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 2 rows, 3 columns and 4 nonzeros
Model fingerprint: 0x9e3d17ea
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+01, 4e+01]
  Bounds range     [2e+01, 4e+02]
  RHS range        [1e+02, 6e+02]
Presolve removed 1 rows and 1 columns
Presolve time: 0.01s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Infeasible model
No optimal solution found.


### branch 2: 23<x2<24

In [23]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model
model = gp.Model("BasicLP")

# Create variables (continuous, with given bounds)
# x1: number of units of Prod1
# x2: number of units of Prod2
# x3: number of units of Prod3
x1 = model.addVar(lb=15.3, ub=102.3, vtype=GRB.CONTINUOUS, name="x1")
x2 = model.addVar(lb=23.2, ub=357.4,  vtype=GRB.CONTINUOUS, name="x2")
x3 = model.addVar(lb=17.5, ub=182.6,  vtype=GRB.CONTINUOUS, name="x3") 


# Set objective function: Maximize total profit
model.setObjective(20*x1 + 40*x2 + 32*x3, GRB.MAXIMIZE)

# Add processing time constraint:
model.addConstr(4.6*x1 + 7.5*x2 + 2.4*x3 <= 556.7, name="ProcessingTime")
model.addConstr(x3 <= 130, name = 'branch 1')
model.addConstr(x2 <= 23, name = 'branch 2')


# Optimize the model
model.optimize()

# Check if an optimal solution was found and print results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(" x1 =", x1.X)
    print(" x2 =", x2.X)
    print(" x3 =", x3.X)
    print(" Objective value =", model.ObjVal)
else:
    print("No optimal solution found.")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 3 rows, 3 columns and 5 nonzeros
Model fingerprint: 0x729e0283
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+01, 4e+01]
  Bounds range     [2e+01, 4e+02]
  RHS range        [2e+01, 6e+02]
Presolve time: 0.01s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Infeasible model
No optimal solution found.


In [25]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model
model = gp.Model("BasicLP")

# Create variables (continuous, with given bounds)
# x1: number of units of Prod1
# x2: number of units of Prod2
# x3: number of units of Prod3
x1 = model.addVar(lb=15.3, ub=102.3, vtype=GRB.CONTINUOUS, name="x1")
x2 = model.addVar(lb=23.2, ub=357.4,  vtype=GRB.CONTINUOUS, name="x2")
x3 = model.addVar(lb=17.5, ub=182.6,  vtype=GRB.CONTINUOUS, name="x3") 


# Set objective function: Maximize total profit
model.setObjective(20*x1 + 40*x2 + 32*x3, GRB.MAXIMIZE)

# Add processing time constraint:
model.addConstr(4.6*x1 + 7.5*x2 + 2.4*x3 <= 556.7, name="ProcessingTime")
model.addConstr(x3 <= 130, name = 'branch 1')
model.addConstr(x2 >= 24, name = 'branch 2')


# Optimize the model
model.optimize()

# Check if an optimal solution was found and print results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(" x1 =", x1.X)
    print(" x2 =", x2.X)
    print(" x3 =", x3.X)
    print(" Objective value =", model.ObjVal)
else:
    print("No optimal solution found.")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 3 rows, 3 columns and 5 nonzeros
Model fingerprint: 0x6e7cc8d4
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+01, 4e+01]
  Bounds range     [2e+01, 4e+02]
  RHS range        [2e+01, 6e+02]
Presolve removed 3 rows and 3 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.3502667e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  5.350266667e+03
Optimal solution found:
 x1 = 15.3
 x2 = 24.0
 x3 = 127.63333333333335
 Objective value = 5350.266666666667


## branch 3

In [29]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model
model = gp.Model("BasicLP")

# Create variables (continuous, with given bounds)
# x1: number of units of Prod1
# x2: number of units of Prod2
# x3: number of units of Prod3
x1 = model.addVar(lb=15.3, ub=102.3, vtype=GRB.CONTINUOUS, name="x1")
x2 = model.addVar(lb=23.2, ub=357.4,  vtype=GRB.CONTINUOUS, name="x2")
x3 = model.addVar(lb=17.5, ub=182.6,  vtype=GRB.CONTINUOUS, name="x3") 


# Set objective function: Maximize total profit
model.setObjective(20*x1 + 40*x2 + 32*x3, GRB.MAXIMIZE)

# Add processing time constraint:
model.addConstr(4.6*x1 + 7.5*x2 + 2.4*x3 <= 556.7, name="ProcessingTime")
model.addConstr(x3 <= 130, name = 'branch 1')
model.addConstr(x2 >= 24, name = 'branch 2')
model.addConstr(x1 >= 16, name = 'branch 3')


# Optimize the model
model.optimize()

# Check if an optimal solution was found and print results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(" x1 =", x1.X)
    print(" x2 =", x2.X)
    print(" x3 =", x3.X)
    print(" Objective value =", model.ObjVal)
else:
    print("No optimal solution found.")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 4 rows, 3 columns and 6 nonzeros
Model fingerprint: 0xfa673f47
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+01, 4e+01]
  Bounds range     [2e+01, 4e+02]
  RHS range        [2e+01, 6e+02]
Presolve removed 4 rows and 3 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.3213333e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  5.321333333e+03
Optimal solution found:
 x1 = 16.0
 x2 = 24.0
 x3 = 126.29166666666669
 Objective value = 5321.333333333334


## branch 4

In [31]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model
model = gp.Model("BasicLP")

# Create variables (continuous, with given bounds)
# x1: number of units of Prod1
# x2: number of units of Prod2
# x3: number of units of Prod3
x1 = model.addVar(lb=15.3, ub=102.3, vtype=GRB.CONTINUOUS, name="x1")
x2 = model.addVar(lb=23.2, ub=357.4,  vtype=GRB.CONTINUOUS, name="x2")
x3 = model.addVar(lb=17.5, ub=182.6,  vtype=GRB.CONTINUOUS, name="x3") 


# Set objective function: Maximize total profit
model.setObjective(20*x1 + 40*x2 + 32*x3, GRB.MAXIMIZE)

# Add processing time constraint:
model.addConstr(4.6*x1 + 7.5*x2 + 2.4*x3 <= 556.7, name="ProcessingTime")
model.addConstr(x3 <= 130, name = 'branch 1')
model.addConstr(x2 >= 24, name = 'branch 2')
model.addConstr(x1 >= 16, name = 'branch 3')
model.addConstr(x3 >= 127, name = 'branch 1')


# Optimize the model
model.optimize()

# Check if an optimal solution was found and print results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(" x1 =", x1.X)
    print(" x2 =", x2.X)
    print(" x3 =", x3.X)
    print(" Objective value =", model.ObjVal)
else:
    print("No optimal solution found.")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 5 rows, 3 columns and 7 nonzeros
Model fingerprint: 0xbc05ed88
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+01, 4e+01]
  Bounds range     [2e+01, 4e+02]
  RHS range        [2e+01, 6e+02]
Presolve removed 4 rows and 0 columns
Presolve time: 0.01s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Infeasible model
No optimal solution found.


In [33]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model
model = gp.Model("BasicLP")

# Create variables 
# x1: number of units of Prod1
# x2: number of units of Prod2
# x3: number of units of Prod3
x1 = model.addVar(lb=15.3, ub=102.3, vtype=GRB.CONTINUOUS, name="x1")
x2 = model.addVar(lb=23.2, ub=357.4,  vtype=GRB.CONTINUOUS, name="x2")
x3 = model.addVar(lb=17.5, ub=182.6,  vtype=GRB.CONTINUOUS, name="x3") 


# Set objective function: Maximize total profit
model.setObjective(20*x1 + 40*x2 + 32*x3, GRB.MAXIMIZE)

# Add processing time constraint:
model.addConstr(4.6*x1 + 7.5*x2 + 2.4*x3 <= 556.7, name="ProcessingTime")
model.addConstr(x3 <= 130, name = 'branch 1')
model.addConstr(x2 >= 24, name = 'branch 2')
model.addConstr(x1 >= 16, name = 'branch 3')
model.addConstr(x3 <= 126, name = 'branch 1')


# Optimize the model
model.optimize()

# Check if an optimal solution was found and print results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(" x1 =", x1.X)
    print(" x2 =", x2.X)
    print(" x3 =", x3.X)
    print(" Objective value =", model.ObjVal)
else:
    print("No optimal solution found.")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 5 rows, 3 columns and 7 nonzeros
Model fingerprint: 0xa2f14915
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+01, 4e+01]
  Bounds range     [2e+01, 4e+02]
  RHS range        [2e+01, 6e+02]
Presolve removed 5 rows and 3 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.3157333e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  5.315733333e+03
Optimal solution found:
 x1 = 16.0
 x2 = 24.093333333333344
 x3 = 126.0
 Objective value = 5315.733333333334


## branch 5

In [39]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model
model = gp.Model("BasicLP")

# Create variables 
# x1: number of units of Prod1
# x2: number of units of Prod2
# x3: number of units of Prod3
x1 = model.addVar(lb=15.3, ub=102.3, vtype=GRB.CONTINUOUS, name="x1")
x2 = model.addVar(lb=23.2, ub=357.4,  vtype=GRB.CONTINUOUS, name="x2")
x3 = model.addVar(lb=17.5, ub=182.6,  vtype=GRB.CONTINUOUS, name="x3") 


# Set objective function: Maximize total profit
model.setObjective(20*x1 + 40*x2 + 32*x3, GRB.MAXIMIZE)

# Add processing time constraint:
model.addConstr(4.6*x1 + 7.5*x2 + 2.4*x3 <= 556.7, name="ProcessingTime")
model.addConstr(x3 <= 130, name = 'branch 1')
model.addConstr(x2 >= 24, name = 'branch 2')
model.addConstr(x1 >= 16, name = 'branch 3')
model.addConstr(x3 <= 126, name = 'branch 4')
model.addConstr(x2<= 24, name = 'branch 4')


# Optimize the model
model.optimize()

# Check if an optimal solution was found and print results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(" x1 =", x1.X)
    print(" x2 =", x2.X)
    print(" x3 =", x3.X)
    print(" Objective value =", model.ObjVal)
else:
    print("No optimal solution found.")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 6 rows, 3 columns and 8 nonzeros
Model fingerprint: 0x6263d0a2
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+01, 4e+01]
  Bounds range     [2e+01, 4e+02]
  RHS range        [2e+01, 6e+02]
Presolve removed 6 rows and 3 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.3150435e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  5.315043478e+03
Optimal solution found:
 x1 = 16.15217391304349
 x2 = 24.0
 x3 = 126.0
 Objective value = 5315.04347826087


In [44]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model
model = gp.Model("BasicLP")

# Create variables 
# x1: number of units of Prod1
# x2: number of units of Prod2
# x3: number of units of Prod3
x1 = model.addVar(lb=15.3, ub=102.3, vtype=GRB.CONTINUOUS, name="x1")
x2 = model.addVar(lb=23.2, ub=357.4,  vtype=GRB.CONTINUOUS, name="x2")
x3 = model.addVar(lb=17.5, ub=182.6,  vtype=GRB.CONTINUOUS, name="x3") 


# Set objective function: Maximize total profit
model.setObjective(20*x1 + 40*x2 + 32*x3, GRB.MAXIMIZE)

# Add processing time constraint:
model.addConstr(4.6*x1 + 7.5*x2 + 2.4*x3 <= 556.7, name="ProcessingTime")
model.addConstr(x3 <= 130, name = 'branch 1')
model.addConstr(x2 >= 24, name = 'branch 2')
model.addConstr(x1 >= 16, name = 'branch 3')
model.addConstr(x3 <= 126, name = 'branch 4')
model.addConstr(x2>= 25, name = 'branch 5')


# Optimize the model
model.optimize()

# Check if an optimal solution was found and print results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(" x1 =", x1.X)
    print(" x2 =", x2.X)
    print(" x3 =", x3.X)
    print(" Objective value =", model.ObjVal)
else:
    print("No optimal solution found.")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 6 rows, 3 columns and 8 nonzeros
Model fingerprint: 0x050233e3
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+01, 4e+01]
  Bounds range     [2e+01, 4e+02]
  RHS range        [2e+01, 6e+02]
Presolve removed 6 rows and 3 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.2613333e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  5.261333333e+03
Optimal solution found:
 x1 = 16.0
 x2 = 25.0
 x3 = 123.16666666666669
 Objective value = 5261.333333333334


## branch 6

In [48]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model
model = gp.Model("BasicLP")

# Create variables 
# x1: number of units of Prod1
# x2: number of units of Prod2
# x3: number of units of Prod3
x1 = model.addVar(lb=15.3, ub=102.3, vtype=GRB.CONTINUOUS, name="x1")
x2 = model.addVar(lb=23.2, ub=357.4,  vtype=GRB.CONTINUOUS, name="x2")
x3 = model.addVar(lb=17.5, ub=182.6,  vtype=GRB.CONTINUOUS, name="x3") 


# Set objective function: Maximize total profit
model.setObjective(20*x1 + 40*x2 + 32*x3, GRB.MAXIMIZE)

# Add processing time constraint:
model.addConstr(4.6*x1 + 7.5*x2 + 2.4*x3 <= 556.7, name="ProcessingTime")
model.addConstr(x3 <= 130, name = 'branch 1')
model.addConstr(x2 >= 24, name = 'branch 2')
model.addConstr(x1 >= 16, name = 'branch 3')
model.addConstr(x3 <= 126, name = 'branch 4')
model.addConstr(x2<= 24, name = 'branch 5')
model.addConstr(x1 >= 17, name = 'branch 6')


# Optimize the model
model.optimize()

# Check if an optimal solution was found and print results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(" x1 =", x1.X)
    print(" x2 =", x2.X)
    print(" x3 =", x3.X)
    print(" Objective value =", model.ObjVal)
else:
    print("No optimal solution found.")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 7 rows, 3 columns and 9 nonzeros
Model fingerprint: 0xe6329083
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+01, 4e+01]
  Bounds range     [2e+01, 4e+02]
  RHS range        [2e+01, 6e+02]
Presolve removed 7 rows and 3 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.2800000e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  5.280000000e+03
Optimal solution found:
 x1 = 17.0
 x2 = 24.0
 x3 = 124.37500000000003
 Objective value = 5280.000000000001


In [46]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model
model = gp.Model("BasicLP")

# Create variables 
# x1: number of units of Prod1
# x2: number of units of Prod2
# x3: number of units of Prod3
x1 = model.addVar(lb=15.3, ub=102.3, vtype=GRB.CONTINUOUS, name="x1")
x2 = model.addVar(lb=23.2, ub=357.4,  vtype=GRB.CONTINUOUS, name="x2")
x3 = model.addVar(lb=17.5, ub=182.6,  vtype=GRB.CONTINUOUS, name="x3") 


# Set objective function: Maximize total profit
model.setObjective(20*x1 + 40*x2 + 32*x3, GRB.MAXIMIZE)

# Add processing time constraint:
model.addConstr(4.6*x1 + 7.5*x2 + 2.4*x3 <= 556.7, name="ProcessingTime")
model.addConstr(x3 <= 130, name = 'branch 1')
model.addConstr(x2 >= 24, name = 'branch 2')
model.addConstr(x1 >= 16, name = 'branch 3')
model.addConstr(x3 <= 126, name = 'branch 4')
model.addConstr(x2<= 24, name = 'branch 5')
model.addConstr(x1 <= 16, name = 'branch 6')


# Optimize the model
model.optimize()

# Check if an optimal solution was found and print results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(" x1 =", x1.X)
    print(" x2 =", x2.X)
    print(" x3 =", x3.X)
    print(" Objective value =", model.ObjVal)
else:
    print("No optimal solution found.")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 7 rows, 3 columns and 9 nonzeros
Model fingerprint: 0xa8fee77d
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+01, 4e+01]
  Bounds range     [2e+01, 4e+02]
  RHS range        [2e+01, 6e+02]
Presolve removed 7 rows and 3 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.3120000e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  5.312000000e+03
Optimal solution found:
 x1 = 16.0
 x2 = 24.0
 x3 = 126.0
 Objective value = 5312.0


In [60]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model
model = gp.Model("BasicLP")

# Create variable
# x1: number of units of Prod1
# x2: number of units of Prod2
# x3: number of units of Prod3
x1 = model.addVar(vtype=GRB.CONTINUOUS, name="x1")
x2 = model.addVar(vtype=GRB.CONTINUOUS, name="x2")
x3 = model.addVar(vtype=GRB.CONTINUOUS, name="x3") 
s1 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s1")
s2 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s2")
s3 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s3")
s4 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s4")
s5 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s5")
s6 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s6")
s7 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s7")

#constraints
model.addConstr(4.6*x1 + 7.5*x2 + 2.4*x3 == 556.7, name="ProcessingTime")
model.addConstr(x1 +s2 == 102.3, name="lower_bound")
model.addConstr(x2 +s3 == 357.4, name="upper_bound")
model.addConstr(x3 +s4 == 182.6, name="lower_bound")
model.addConstr(x1 -s5 == 15.3, name="upper_bound")
model.addConstr(x2 -s6 == 23.2, name="lower_bound")
model.addConstr(x3 -s7 == 17.5, name="upper_bound")



# Set objective function: Maximize total profit
model.setObjective(20*x1 + 40*x2 + 32*x3 + 0*s1 + 0*s2+ 0*s3+ 0*s4+ 0*s5+ 0*s6+ 0*s7, GRB.MAXIMIZE)


# Optimize the model
model.optimize()

# Check if an optimal solution was found and print results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(" x1 =", x1.X)
    print(" x2 =", x2.X)
    print(" x3 =", x3.X)
    print(" Objective value =", model.ObjVal)
else:
    print("No optimal solution found.")


Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 7 rows, 10 columns and 15 nonzeros
Model fingerprint: 0x75b94c1f
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+01, 4e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+01, 6e+02]
Presolve removed 7 rows and 10 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.3982667e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  5.398266667e+03
Optimal solution found:
 x1 = 15.3
 x2 = 23.2
 x3 = 130.13333333333335
 Objective value = 5398.266666666667


In [78]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np


# Helper function: print a NumPy matrix in LaTeX bmatrix form
def latex_matrix(mat, name="Matrix", decimals=3, tol=1e-3):
    """
    Prints a matrix in LaTeX bmatrix form, setting any values
    with absolute value < tol to 0. 
    """
    # Round small values to 0
    mat_rounded = np.where(np.abs(mat) < tol, 0, mat)

    # Build rows as strings like "1.000 & 2.000 & 3.000"
    rows_str = []
    for row in mat_rounded:
        row_str = " & ".join(f"{val:.{decimals}f}" for val in row)
        rows_str.append(row_str)

    # Join rows with LaTeX line breaks
    matrix_body = " \\\\\n".join(rows_str)

    # Wrap in LaTeX bmatrix environment
    latex_str = f"\\[\n{name} = \\begin{{bmatrix}}\n{matrix_body}\n\\end{{bmatrix}}\n\\]"

    print(latex_str)
    print()  # extra blank line for readability

# Create a new model
model = gp.Model("BasicLP")

# Create variables (x1, x2, x3 and slack variables s1...s7)
x1 = model.addVar(vtype=GRB.CONTINUOUS, name="x1")
x2 = model.addVar(vtype=GRB.CONTINUOUS, name="x2")
x3 = model.addVar(vtype=GRB.CONTINUOUS, name="x3") 
s1 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s1")
s2 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s2")
s3 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s3")
s4 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s4")
s5 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s5")
s6 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s6")
s7 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s7")

# Add constraints
model.addConstr(4.6*x1 + 7.5*x2 + 2.4*x3 == 556.7, name="ProcessingTime")
model.addConstr(x1 + s2 == 102.3, name="constr2")
model.addConstr(x2 + s3 == 357.4, name="constr3")
model.addConstr(x3 + s4 == 182.6, name="constr4")
model.addConstr(x1 - s5 == 15.3, name="constr5")
model.addConstr(x2 - s6 == 23.2, name="constr6")
model.addConstr(x3 - s7 == 17.5, name="constr7")

# Set objective function: Maximize total profit
model.setObjective(20*x1 + 40*x2 + 32*x3, GRB.MAXIMIZE)

# Optimize the model
model.optimize()

if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(" x1 =", x1.X)
    print(" x2 =", x2.X)
    print(" x3 =", x3.X)
    print(" Objective value =", model.ObjVal)

    # ----- Extract the constraint matrix A -----
    # Get the constraints (rows) and the variables (columns)
    constrs = model.getConstrs()
    vars_list = model.getVars()
    num_rows = len(constrs)
    num_vars = len(vars_list)

    # Create a dictionary to map each variable to its index in vars_list
    var_index = {var.VarName: idx for idx, var in enumerate(vars_list)}

    # Initialize A as a numpy array
    A = np.zeros((num_rows, num_vars))

    # Loop over each constraint and fill in the coefficients
    for i, constr in enumerate(constrs):
        row = model.getRow(constr)  # This returns a LinExpr
        for j in range(row.size()):
            var = row.getVar(j)
            coef = row.getCoeff(j)
            # Look up the index of the variable using its name
            idx = var_index[var.VarName]
            A[i, idx] = coef

    print("\nConstraint matrix A:")
    print(A)

    # ----- Identify the basic variables using the VBasis attribute -----
    # VBasis: 0 indicates basic, -1 nonbasic at lower bound,
    # and -2 nonbasic at upper bound.
    vbasis = model.getAttr("VBasis", vars_list)
    basic_indices = [i for i, b in enumerate(vbasis) if b == 0]

    print("\nIndices of basic variables:", basic_indices)

    # Extract the basis matrix B: columns of A corresponding to basic variables.
    B = A[:, basic_indices]
    print("\nBasis matrix B:")
    print(B)

    # Compute the inverse of B (if B is square and invertible)
    try:
        B_inv = np.linalg.inv(B)
        tol = 1e-3
        B_inv_rounded = np.where(np.abs(B_inv) < tol, 0, B_inv)
        print("B_inv: ")
        print(B_inv_rounded)

        
        P = B_inv @ A
        # Apply the same tolerance-based rounding
        P_rounded = np.where(np.abs(P) < tol, 0, P)
        print("Product of B_inv and A:")
        print(P_rounded)

        #Compute the product B_inv * b (RHS vector):
        b_values = model.getAttr("RHS", constrs)  # returns a list of RHS values
        b = np.array(b_values)
        Pb = B_inv @ b
        Pb_rounded = np.where(np.abs(Pb) < tol, 0, Pb)
        print("\nProduct of B_inv and b:")
        print(Pb_rounded)

   
    except np.linalg.LinAlgError:
        print("\nBasis matrix B is singular and cannot be inverted.")
else:
    print("No optimal solution found.")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 7 rows, 10 columns and 15 nonzeros
Model fingerprint: 0x75b94c1f
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+01, 4e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+01, 6e+02]
Presolve removed 7 rows and 10 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.3982667e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  5.398266667e+03
Optimal solution found:
 x1 = 15.3
 x2 = 23.2
 x3 = 130.13333333333335
 Objective value = 5398.266666666667

Constraint matrix A:
[[ 4.6  7.5  2.4  0.   0.   0.   0.   0.   0.   0. ]
 [ 1.   0.   0.   0.   1.   0.   0.   0.   0.   0. ]
 [ 0.   1.   0

In [106]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
from tabulate import tabulate

def tabulate_matrix(mat, name="Matrix", decimals=3, tol=1e-3):
    """
    Prints a NumPy array in a neat table using the `tabulate` library.
    Values with absolute value < tol are set to 0 for readability.
    """
    # Zero out small values
    mat_rounded = np.where(np.abs(mat) < tol, 0, mat)

    # Convert each row to a list of strings with desired decimal format
    table_data = []
    for row in mat_rounded:
        row_str = [f"{val:.{decimals}f}" for val in row]
        table_data.append(row_str)

    print(f"\n{name}:")
    print(tabulate(table_data, tablefmt="fancy_grid"))

# -----------------------------
# Main Gurobi model code
# -----------------------------
model = gp.Model("BasicLP")

# Create variables (x1, x2, x3 and slack variables s1...s7)
x1 = model.addVar(vtype=GRB.CONTINUOUS, name="x1")
x2 = model.addVar(vtype=GRB.CONTINUOUS, name="x2")
x3 = model.addVar(vtype=GRB.CONTINUOUS, name="x3") 
s1 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s1")
s2 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s2")
s3 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s3")
s4 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s4")
s5 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s5")
s6 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s6")
s7 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s7")

# Add constraints
model.addConstr(4.6*x1 + 7.5*x2 + 2.4*x3 +s1 == 556.7, name="ProcessingTime")
model.addConstr(x1 + s2 == 102.3, name="constr2")
model.addConstr(x2 + s3 == 357.4, name="constr3")
model.addConstr(x3 + s4 == 182.6, name="constr4")
model.addConstr(x1 - s5 == 15.3, name="constr5")
model.addConstr(x2 - s6 == 23.2, name="constr6")
model.addConstr(x3 - s7 == 17.5, name="constr7")

# Set objective function: Maximize total profit
model.setObjective(20*x1 + 40*x2 + 32*x3, GRB.MAXIMIZE)

# Optimize the model
model.optimize()

if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(f" x1 = {x1.X}")
    print(f" x2 = {x2.X}")
    print(f" x3 = {x3.X}")
    print(" Objective value =", model.ObjVal)

    # 1) Extract the constraint matrix A
    constrs = model.getConstrs()
    vars_list = model.getVars()
    num_rows = len(constrs)
    num_vars = len(vars_list)

    var_index = {var.VarName: idx for idx, var in enumerate(vars_list)}
    A = np.zeros((num_rows, num_vars))

    for i, constr in enumerate(constrs):
        row_expr = model.getRow(constr)
        for j in range(row_expr.size()):
            var = row_expr.getVar(j)
            coef = row_expr.getCoeff(j)
            idx = var_index[var.VarName]
            A[i, idx] = coef

    tabulate_matrix(A, name="Constraint matrix A")

    # 2) Identify basic variables, form B
    vbasis = model.getAttr("VBasis", vars_list)
    basic_indices = [i for i, b in enumerate(vbasis) if b == 0]

    B = A[:, basic_indices]
    tabulate_matrix(B, name="Basis matrix B")

    # 3) Compute B_inv, B_inv*A, B_inv*b
    try:
        B_inv = np.linalg.inv(B)
        tabulate_matrix(B_inv, name="Inverse of B")

        BA = B_inv @ A
            # 3) Compute B_inv, B_inv*A, B_inv*b
    
        col_headers = [var.VarName for var in vars_list]  # column headers

        decimals = 5
        tol = 1e-5
        BA_rounded = np.where(np.abs(BA) < tol, 0, BA)

        table_data = []
        for row in BA_rounded:
            row_str = [f"{val:.{decimals}f}" for val in row]
            table_data.append(row_str)

        # Print the fancy table with headers
        print(tabulate(table_data, headers=col_headers, tablefmt="fancy_grid"))

        # Now handle B_inv * b the same way as before
        b_values = model.getAttr("RHS", constrs)
        b = np.array(b_values)
        Bb = B_inv @ b
        Bb_col = Bb.reshape(-1,1)
        tabulate_matrix(Bb_col, name="Product of B_inv and b")

    except np.linalg.LinAlgError:
        print("\nBasis matrix B is singular and cannot be inverted.")

        # RHS vector
        b_values = model.getAttr("RHS", constrs)
        b = np.array(b_values)

        Bb = B_inv @ b
        # Reshape as a column vector so tabulate_matrix prints it nicely
        Bb_col = Bb.reshape(-1,1)
        tabulate_matrix(Bb_col, name="Product of B_inv and b")

    except np.linalg.LinAlgError:
        print("\nBasis matrix B is singular and cannot be inverted.")

else:
    print("No optimal solution found.")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 7 rows, 10 columns and 16 nonzeros
Model fingerprint: 0x247fc598
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+01, 4e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+01, 6e+02]
Presolve removed 7 rows and 10 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.3982667e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  5.398266667e+03
Optimal solution found:
 x1 = 15.3
 x2 = 23.2
 x3 = 130.13333333333335
 Objective value = 5398.266666666667

Constraint matrix A:
╒═════╤═════╤═════╤═══╤═══╤═══╤═══╤════╤════╤════╕
│ 4.6 │ 7.5 │ 2.4 │ 1 │ 0 │ 0 │ 0 │  0 │  0 │  0 │
├─────┼─────┼─────

#### so with the product of B_inv and A, and B_inv and b, we can formulate the cuts.

In [138]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
from tabulate import tabulate

def tabulate_matrix(mat, name="Matrix", decimals=3, tol=1e-3):
    # Zero out small values
    mat_rounded = np.where(np.abs(mat) < tol, 0, mat)

    # Convert each row to a list of strings with desired decimal format
    table_data = []
    for row in mat_rounded:
        row_str = [f"{val:.{decimals}f}" for val in row]
        table_data.append(row_str)

    print(f"\n{name}:")
    print(tabulate(table_data, tablefmt="fancy_grid"))

# -----------------------------
# Main Gurobi model code
# -----------------------------
model = gp.Model("BasicLP")

# Create variables (x1, x2, x3 and slack variables s1...s7)
x1 = model.addVar(vtype=GRB.CONTINUOUS, name="x1")
x2 = model.addVar(vtype=GRB.CONTINUOUS, name="x2")
x3 = model.addVar(vtype=GRB.CONTINUOUS, name="x3") 
s1 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s1")
s2 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s2")
s3 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s3")
s4 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s4")
s5 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s5")
s6 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s6")
s7 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s7")
s8 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s8")
s9 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s9")
s10 = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="s10")

# Add constraints
model.addConstr(4.6*x1 + 7.5*x2 + 2.4*x3 + s1 == 556.7, name="ProcessingTime")
model.addConstr(x1 + s2 == 102.3, name="constr2")
model.addConstr(x2 + s3 == 357.4, name="constr3")
model.addConstr(x3 + s4 == 182.6, name="constr4")
model.addConstr(x1 - s5 == 15.3, name="constr5")
model.addConstr(x2 - s6 == 23.2, name="constr6")
model.addConstr(x3 - s7 == 17.5, name="constr7")

#we add the gomory cuts
model.addConstr(0.416*(556.7-4.6*x1-7.5*x2-2.4*x3) +0.916*(x1-15.3) + 0.1256*(x2-23.2) -s8 == 0.633, name="gomory cut 1")
model.addConstr(0.583*(556.7-4.6*x1-7.5*x2-2.4*x3) +0.08345*(x1-15.3) + 0.8744*(x2-23.2) -s9 == 0.464, name="gomory cut 2")
model.addConstr(0.416*(556.7-4.6*x1-7.5*x2-2.4*x3) +0.916*(x1-15.3) + 0.1256*(x2-23.2) -s10 == 0.133, name="gomory cut 3")



# Set objective function: Maximize total profit
model.setObjective(20*x1 + 40*x2 + 32*x3, GRB.MAXIMIZE)

# Optimize the model
model.optimize()

if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(f" x1 = {x1.X}")
    print(f" x2 = {x2.X}")
    print(f" x3 = {x3.X}")
    print(" Objective value =", model.ObjVal)

    # 1) Extract the constraint matrix A
    constrs = model.getConstrs()
    vars_list = model.getVars()
    num_rows = len(constrs)
    num_vars = len(vars_list)

    var_index = {var.VarName: idx for idx, var in enumerate(vars_list)}
    A = np.zeros((num_rows, num_vars))

    for i, constr in enumerate(constrs):
        row_expr = model.getRow(constr)
        for j in range(row_expr.size()):
            var = row_expr.getVar(j)
            coef = row_expr.getCoeff(j)
            idx = var_index[var.VarName]
            A[i, idx] = coef

    tabulate_matrix(A, name="Constraint matrix A")

    # 2) Identify basic variables, form B
    vbasis = model.getAttr("VBasis", vars_list)
    basic_indices = [i for i, b in enumerate(vbasis) if b == 0]

    B = A[:, basic_indices]
    tabulate_matrix(B, name="Basis matrix B")

    # 3) Compute B_inv, B_inv*A, B_inv*b
    try:
        B_inv = np.linalg.inv(B)
        tabulate_matrix(B_inv, name="Inverse of B")

        BA = B_inv @ A
            # 3) Compute B_inv, B_inv*A, B_inv*b
    
        col_headers = [var.VarName for var in vars_list]  # column headers

        decimals = 5
        tol = 1e-5
        BA_rounded = np.where(np.abs(BA) < tol, 0, BA)

        table_data = []
        for row in BA_rounded:
            row_str = [f"{val:.{decimals}f}" for val in row]
            table_data.append(row_str)

        # Print the fancy table with headers
        print("Product of B_inv and A:")
        print(tabulate(table_data, headers=col_headers, tablefmt="fancy_grid"))

        # Now handle B_inv * b the same way as before
        b_values = model.getAttr("RHS", constrs)
        b = np.array(b_values)
        Bb = B_inv @ b
        Bb_col = Bb.reshape(-1,1)
        tabulate_matrix(Bb_col, name="Product of B_inv and b")

    except np.linalg.LinAlgError:
        print("\nBasis matrix B is singular and cannot be inverted.")

        # RHS vector
        b_values = model.getAttr("RHS", constrs)
        b = np.array(b_values)

        Bb = B_inv @ b
        # Reshape as a column vector so tabulate_matrix prints it nicely
        Bb_col = Bb.reshape(-1,1)
        tabulate_matrix(Bb_col, name="Product of B_inv and b")

    except np.linalg.LinAlgError:
        print("\nBasis matrix B is singular and cannot be inverted.")

else:
    print("No optimal solution found.")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 10 rows, 13 columns and 28 nonzeros
Model fingerprint: 0xf145827e
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+01, 4e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+01, 6e+02]
Presolve removed 7 rows and 10 columns
Presolve time: 0.00s
Presolved: 3 rows, 3 columns, 9 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.3982667e+03   2.630038e-01   0.000000e+00      0s
       5    5.3779782e+03   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.00 seconds (0.00 work units)
Optimal objective  5.377978205e+03
Optimal solution found:
 x1 = 15.3
 x2 = 23.2
 x3 = 129.4993189102564
 Objective value = 5377.9782051282045

Constraint matrix A:
╒════════╤════════╤════════╤═══╤═══╤═══╤═══╤════╤═══